<a href="https://colab.research.google.com/github/Shreyojit/News-Article-NER_and_KG_Extracting_Subject_Verb_Action_And_KG/blob/main/News_Article_NER_and_KG_Extracting_Subject_Verb_Action.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What is BERT?

BERT stands for **Bidirectional Encoder Representations from Transformers**. The name itself gives us several clues about what BERT is all about.

BERT architecture consists of several Transformer encoders stacked together. Each Transformer encoder encapsulates two sub-layers: a **self-attention layer** and a **feed-forward layer**.  
There are two different BERT models:

- **BERT base**: Consists of 12 layers of Transformer encoder, 12 attention heads, 768 hidden size, and 110M parameters.
- **BERT large**: Consists of 24 layers of Transformer encoder, 16 attention heads, 1024 hidden size, and 340M parameters.

## BERT Input and Output

The BERT model expects a sequence of tokens (words) as an input. In each sequence of tokens, there are two special tokens that BERT expects:

- **[CLS]**: This is the first token of every sequence, which stands for classification token.
- **[SEP]**: This token helps BERT understand which token belongs to which sequence. It is important for tasks like next sentence prediction or question answering.  
  If there is only one sequence, this token is appended to the end of the sequence.

### Additional Notes:

1. The maximum size of tokens that can be fed into the BERT model is **512**.
   - If the tokens in a sequence are fewer than 512, we use padding to fill the unused slots with the `[PAD]` token.
   - If the tokens in a sequence exceed 512, truncation is needed.

2. **Output:**  
   BERT outputs an **embedding vector of size 768** for each token. These vectors can be used for various NLP tasks such as:
   - Text classification
   - Next sentence prediction
   - Named-Entity Recognition (NER)
   - Question answering

   For a text classification task, we focus on the embedding vector from the special `[CLS]` token. This vector of size 768 is used as input for the classifier, which outputs a vector of size equal to the number of classes in the classification task.

---

### Illustration of BERT Architecture

![BERT Architecture](https://www.researchgate.net/publication/349546860/figure/fig2/AS:994573320994818@1614136166736/The-Transformer-based-BERT-base-architecture-with-twelve-encoder-blocks.ppm)


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/News-Article-NER_and_KG/all-the-news.csv'

# Read only the first 10,000 rows (adjust the number as needed)
df = pd.read_csv(file_path, nrows=1000000)

# Display the first 5 rows
print(df.head())


   Unnamed: 0.1  Unnamed: 0                 date  year  month  day  \
0             0           0  2016-12-09 18:31:00  2016   12.0    9   
1             1           1  2016-10-07 21:26:46  2016   10.0    7   
2             2           2  2018-01-26 00:00:00  2018    1.0   26   
3             3           3  2019-06-27 00:00:00  2019    6.0   27   
4             4           4  2016-01-27 00:00:00  2016    1.0   27   

        author                                              title  \
0  Lee Drutman  We should take concerns about the health of li...   
1  Scott Davis  Colts GM Ryan Grigson says Andrew Luck's contr...   
2          NaN       Trump denies report he ordered Mueller fired   
3          NaN  France's Sarkozy reveals his 'Passions' but in...   
4          NaN  Paris Hilton: Woman In Black For Uncle Monty's...   

                                             article  \
0  This post is part of Polyarchy, an independent...   
1   The Indianapolis Colts made Andrew Luck the h...

In [4]:
df.shape

(1000000, 12)

In [5]:
sources = df["publication"].unique()
print(sources)

['Vox' 'Business Insider' 'Reuters' 'TMZ' 'Vice' 'Vice News'
 'Hyperallergic' 'TechCrunch' 'Axios' 'Refinery 29' 'The Verge' 'Mashable'
 'People' 'Economist' 'CNN' 'Gizmodo']


In [6]:
condition = df["publication"].isin(["CNN", "Economist"])

content_df = df.loc[condition, :]["article"][:1000]
# Remove any rows with NaN values
content_df = content_df.dropna()

content_df.shape


(906,)

In [7]:
content_df.head()

,article
640996,A NUMBER of economic trends that have been sim...
640997,THE rise of big emerging economies like China ...
640999,PFIZER has always prided itself on its commitm...
641006,THE Ben Abeba restaurant is a spiral-shaped co...
641008,IT IS more than two weeks since the Federal Re...


In [8]:
for article in content_df[:2]:
    print(article)

A NUMBER of economic trends that have been simmering for years will come to full boil in 2016—and result in some striking statistics. One example is in wealth inequality. The share of wealth of the richest 1% of the world's population will, according to Oxfam, exceed that of the remaining 99% of people. Yet there is also some good news for ordinary folk: in at least one respect private investing is becoming more democratic. The amount of global crowdfunding investment is expected to reach $34 billion in 2015, and in 2016 will surpass the annual money committed by investors to venture-capital funds globally.
THE rise of big emerging economies like China and India, and the steady march of globalisation, have led to a surge in the numbers of people wanting to travel abroad for business or tourism. As a result, demand for visas is at unprecedented levels. In the fiscal year to the end of September 2014 the United States granted just under 10m visas—up from around 6m in 1997, despite blips 

In [9]:
!pip install spacy


In [10]:
!python -m spacy download en_core_web_md


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 19.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [11]:
import spacy
nlp = spacy.load('en_core_web_md')  # or 'en_core_web_lg'


In [12]:


text = "He would not tell the police what he had learned"

doc = nlp(text)

for token in doc:
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_, token.shape_, token.is_alpha, token.is_stop  )


He he PRON PRP nsubj Xx True True
would would AUX MD aux xxxx True True
not not PART RB neg xxx True True
tell tell VERB VB ROOT xxxx True False
the the DET DT det xxx True True
police police NOUN NNS dobj xxxx True False
what what PRON WP dobj xxxx True True
he he PRON PRP nsubj xx True True
had have AUX VBD aux xxx True True
learned learn VERB VBN ccomp xxxx True False



### In below the named_entities variable should show me a dict structure like below

```python
{
    'GPE': {
        'WASHINGTON': 1,
        'Obama': 1,
        'the District of Columbia Circuit': 1,
        'Manhattan': 13,
        'New York City': 3,
        'Bronx': 19
    },
    'NORP': {
        'Republicans': 15,
        'Americans': 1,
        'Republican': 2,
        'Hispanic': 1
    }
}


In [13]:
def return_entities_and_processed_docs(data_frame):
    """
    Extracts named entities from a DataFrame and returns them along with the processed docs.

    Args:
        data_frame (DataFrame): The DataFrame from which to extract named entities.

    Returns:
        dict: A dictionary mapping entity types to a dictionary of entity names to counts.
        list: A list of processed docs.
    """
    # Initialize the dictionary to hold named entities
    named_entities = {}
    # Initialize the list to hold processed docs
    processed_docs = []

    # Process each item in the DataFrame
    for item in data_frame:
        # Process the item with a language model
        doc = nlp(item)
        # Add the processed doc to the list
        processed_docs.append(doc)

        # For each named entity in the doc...
        for ent in doc.ents:
            # Extract the entity text (e.g., 'WASHINGTON')
            entity_text = ent.text
            # Extract the entity type (e.g., 'GPE')
            entity_type = str(ent.label_)
            # Initialize a dictionary to hold the current entities
            current_ents = {}

            # If the entity type is already in the named entities dictionary...
            if entity_type in named_entities.keys():
                # Get the dictionary of entity names to counts
                current_ents = named_entities.get(entity_type)

            # Increment the count for the entity name
            # This will add 1 to the count in the inner dictionary
            current_ents[entity_text] = current_ents.get(entity_text, 0) + 1

            # Update the inner dictionary for the entity type
            named_entities[entity_type] = current_ents

    # Return the named entities and the processed docs
    return named_entities, processed_docs


named_entities, processed_docs = return_entities_and_processed_docs(content_df)

In [14]:
named_entities


{'DATE': {'years': 109,
  '2016': 134,
  '2015': 228,
  'annual': 88,
  'the fiscal year': 2,
  'the end of September 2014': 1,
  '1997': 19,
  'September 11th 2001': 2,
  '2007-08': 13,
  'several months’': 1,
  'the past ten years': 5,
  'the previous five years': 1,
  '2001': 30,
  '2013': 162,
  '1906': 3,
  '2014': 293,
  'a year': 30,
  'January': 63,
  'between 2002 and 2011': 1,
  '2012': 151,
  'two years': 26,
  '2011': 105,
  'the same year': 2,
  '12th-century': 1,
  'a few years ago': 5,
  'two days': 12,
  'days': 14,
  '21 years': 4,
  'daily': 37,
  '2000': 35,
  'a year each year': 1,
  '64': 1,
  'monthly': 11,
  'today': 189,
  'a decade': 28,
  '1991': 11,
  '2005': 46,
  '2010': 100,
  'May 2015': 2,
  'September of 2015': 1,
  'five years': 32,
  'more than two weeks': 2,
  'over nine years': 1,
  'recent years': 63,
  'a rosy year': 1,
  'the end of the year': 10,
  '1930s': 5,
  'the late 1990s': 5,
  'the 1930s': 7,
  '1937': 3,
  'THE autumn': 1,
  '1962': 3,


In [15]:
len(processed_docs)

906

#
Better way to print the Entity Types and Values

In this code, you print out the type of a named entity (e.g., ORG) and for each type, you extract all entities assigned with this type in the dictionary, sorted by their frequency in descending order.

Print out the most frequent 10 entities per type


In [16]:
def print_top_10(named_entities):
    for key in named_entities.keys():
        print(key)
        entities = named_entities.get(key)

        # Sort the entries by their frequency in descending
        # order and print out the most frequent n ones
        sorted_keys = sorted(entities, key=entities.get, reverse=True )
        for item in sorted_keys[:10]:
            if (entities.get(item) > 1 ):
                print(" " + item + ": " + str(entities.get(item)))

print_top_10(named_entities)

DATE
 2014: 293
 2015: 228
 last year: 222
 today: 189
 2013: 162
 2012: 151
 December: 138
 2016: 134
 this year: 133
 this week: 120
CARDINAL
 one: 782
 two: 613
 three: 266
 One: 265
 four: 157
 five: 104
 half: 93
 six: 73
 millions: 63
 ten: 60
PERCENT
 10%: 58
 20%: 36
 5%: 34
 25%: 31
 40%: 30
 7%: 26
 50%: 24
 2%: 23
 15%: 23
 30%: 22
ORG
 EU: 461
 Congress: 94
 UN: 91
 Fed: 80
 IMF: 77
 Islamic State: 76
 Apple: 71
 Facebook: 70
 Google: 64
 Labour: 55
MONEY
 $1 billion: 13
 30: 10
 $5 billion: 9
 35: 8
 100: 8
 $100 billion: 6
 $3 billion: 6
 18.99: 6
 $1.8 billion: 6
 28: 5
GPE
 America: 868
 China: 864
 Britain: 566
 Turkey: 323
 London: 287
 Russia: 253
 Syria: 205
 Germany: 195
 Iran: 176
 India: 172
QUANTITY
 1m: 8
 1.2m: 5
 3m: 5
 more than: 4
 4m: 4
 1.1m: 4
 80m: 4
 2m: 4
 500m: 4
 6m: 3
NORP
 American: 582
 Chinese: 357
 British: 304
 European: 241
 Republican: 221
 French: 157
 Americans: 145
 Russian: 145
 Turkish: 137
 Italian: 111
PERSON
 Trump: 296
 Obama: 174
 

In [17]:
named_entities.keys()

dict_keys(['DATE', 'CARDINAL', 'PERCENT', 'ORG', 'MONEY', 'GPE', 'QUANTITY', 'NORP', 'PERSON', 'TIME', 'WORK_OF_ART', 'LOC', 'ORDINAL', 'LAW', 'EVENT', 'LANGUAGE', 'FAC', 'PRODUCT'])

In [18]:
def calculate_entity_span(document, entity):
    """
    Calculate the span of a named entity in a document.

    Args:
        document (spacy.tokens.Doc): The document to search for the entity.
        entity (str): The entity to find in the document.

    Returns:
        list: A list of indices representing the span of the entity in the document.
    """
    # Initialize a list to hold the indices
    indexes = []

    # Iterate over the entities in the document
    for ent in document.ents:
        # If the entity text matches the entity we're looking for...
        if ent.text == entity:
            # Iterate over the range from the entity's start to its end
            for i in range(int(ent.start), int(ent.end)):
                # Append the index to the list
                indexes.append(i)

    # Return the list of indices
    return indexes


In [19]:
entity = "The New York Times"

sentences = "The New York Times wrote about Apple"

doc = nlp(sentences)

calculate_entity_span(doc, entity)

[0, 1, 2, 3]

In [20]:
sentences = ["The New York Times wrote about Apple"]

for sentence in sentences:
    doc = nlp(sentence)
    for token in doc:
        print(token.dep_)

det
compound
compound
nsubj
ROOT
prep
pobj


In [21]:
def calc_entity_subject_object(document, entity, indexes):
    """
    Analyze a document to find actions related to a specific entity.

    Args:
        document (spacy.tokens.Doc): The document to analyze.
        entity (str): The entity to find actions for.
        indexes (list): A list of indices in the document where the entity appears.

    Returns:
        None: Prints out the sentence and all actions involving the entity.
    """
    actions = []
    action = ''
    participant1 = ''
    participant2 = ''

    for token in document:
        # Next, you identify the main verb expressing the main action in the sentence
        # To extract the relation, we have to find the ROOT of the sentence (which is also the verb of the sentence)
        if token.pos_ == "VERB" and token.dep_ == 'ROOT':
            # Initialize the indexes for the subject and the object related to the main verb
            subj_ind = -1
            obj_ind = -1
            # Store the main verb itself (token.text) in the action variable
            action = token.text
            children = [child for child in token.children ]
            for child1 in children:
                # Find the subject via the nsubj relation and store it as participant1
                # and its index as subj_ind
                if child1.dep_ == 'nsubj':
                    participant1 = child1.text
                    subj_ind = int(child1.i)
                # If there is a preposition attached to the verb (e.g., “write about”), then
                # you need to search for the indirect object as the second participant.
                if child1.dep_ == 'prep':
                    participant2 = ''
                    child1_children = [child for child in child1.children]
                    for child2 in child1_children:
                        # If such an object is a noun or a proper noun,
                        # you store it as participant2 and its index as obj_ind
                        if child2.pos_ == 'NOUN' or child2.pos_ == 'PROPN':
                            participant2 = child2.text
                            obj_ind = int(child2.i)

                    # If at this point both participants of the main action have been identified and
                    # their indexes are included in the indexes of the words covered by the entity,
                    # you add the action with two participants to the list of actions.
                    if not participant2=="":
                        if subj_ind in indexes:
                            actions.append(entity + " " + action + " " + child1.text + " " + participant2)
                        elif obj_ind in indexes:
                            actions.append(participant1 + " " + action + " " + child1.text + " " + entity)

                # Otherwise, if there is no preposition attached to the verb,
                # participant2 is a direct object of the main verb,
                # which can be identified via the dobj relation
                if child1.dep_ == 'dobj' and (child1.pos_ == 'NOUN' or child1.pos_ == 'PROPN' ):
                    participant2 = child1.text
                    obj_ind = int(child1.i)
                    # In this case, you apply the same strategy as above,
                    # adding the action with two participants to the list of actions.
                    if subj_ind in indexes:
                         actions.append(entity + " " + action + " " + participant2)
                    elif obj_ind in indexes:
                        actions.append(participant1 + " " + action + " " + entity)
    # Finally if the final list of actions is not empty,
    # Print out the sentence and all actions together with the participants.
    if not len(actions) == 0:
        print(f"\nSentence = {document}")
        for item in actions:
            print(item)

In [22]:
def return_docs_of_given_ent_type(processed_docs, entity, ent_type):
    """
    Extracts sentences from a list of processed documents that contain a given named entity of a specific type.

    Args:
        processed_docs (list): A list of processed documents. Each document is a spacy.tokens.Doc object.
        entity (str): The named entity to search for.
        ent_type (str): The type of the named entity (e.g. 'ORG', 'PERSON', 'GPE')

    Returns:
        output_sentences (list): A list of sentences containing the named entity of the specified type.
    """
    output_sentences = []
    for doc in processed_docs:
        for sentence in doc.sents:
            # Only consider sentences that contain the input entity
            # of the specified type among its named entities
            if entity in [ent.text for ent in sentence.ents if ent.label_ == ent_type ]:
                output_sentences.append(sentence)
    return output_sentences

entity = "Apple"

ent_sentences = return_docs_of_given_ent_type(processed_docs, entity, 'ORG' )
print(ent_sentences)

[Another popular Japanese product offering the illusion of personal space is the Solo Theatre (pictured), a cardboard box that users put over their heads, which has a slot for Apple’s iPhone., It was also revealed at CES that Toyota would adopt Ford’s in-car technology, which is a competitor to Apple’s CarPlay and Google’s Android Auto, to access smartphone apps and other features., Apple, which is said to be planning an electric car, may try to have them made in the same way as it does its iPhones, outsourcing to a contract manufacturer., The bright spot was Apple, which this week also said that 10m people have signed up to the music-streaming service it launched six months ago., The biggest companies are building awe-inspiring headquarters: Apple’s “spaceship”, designed by Norman Foster, will cover 2.8m square feet (260,000 square metres) and Google’s new offices will sit under a vast, translucent dome., Apple’s iTunes store, historically the digital marketplace’s biggest music stall

In [23]:
for sentence in ent_sentences:
    indexes = calculate_entity_span(sentence, entity)
    calc_entity_subject_object(sentence, entity, indexes)


Sentence = Treading directly on the toes of Pandora, Apple will also offer individuals the chance to “create [their] own stations” based on their preferences.
Apple offer chance

Sentence = In 2014 Apple put a podcast app on the iPhone that is all but indelible.
Apple put app
Apple put on iPhone

Sentence = Nevertheless, Apple can now boast a new landmark: one billion active devices over the past 90 days.
Apple boast landmark
Apple boast over days

Sentence = Apple has also added haptic effects with something it calls a “taptic engine”, in effect a refined tiny vibrator which provides subtle taps in response to certain finger movements.
Apple added effects
Apple added vibrator

Sentence = On January 26th Apple announced profits for its most recent quarter of $18.4 billion, more than any listed firm worldwide has yet made in a three-month period.
Apple announced profits
Apple announced for quarter

Sentence = In its rise to greatness, Apple has repeatedly shrugged off bouts of panic am

In [24]:
def return_docs_of_given_ent_type(processed_docs, entity, ent_type):
    """
    Extracts sentences from a list of processed documents that contain a given named entity of a specific type.

    Args:
        processed_docs (list): A list of processed documents. Each document is a spacy.tokens.Doc object.
        entity (str): The named entity to search for.
        ent_type (str): The type of the named entity (e.g. 'ORG', 'PERSON', 'GPE')

    Returns:
        output_sentences (list): A list of sentences containing the named entity of the specified type.
    """
    output_sentences = []
    for doc in processed_docs:
        for sentence in doc.sents:
            # Only consider sentences that contain the input entity
            # of the specified type among its named entities
            if entity in [ent.text for ent in sentence.ents if ent.label_ == ent_type ]:
                output_sentences.append(sentence)
    return output_sentences

entity = "Apple"

ent_sentences = return_docs_of_given_ent_type(processed_docs, entity, 'ORG' )
print(ent_sentences)

[Another popular Japanese product offering the illusion of personal space is the Solo Theatre (pictured), a cardboard box that users put over their heads, which has a slot for Apple’s iPhone., It was also revealed at CES that Toyota would adopt Ford’s in-car technology, which is a competitor to Apple’s CarPlay and Google’s Android Auto, to access smartphone apps and other features., Apple, which is said to be planning an electric car, may try to have them made in the same way as it does its iPhones, outsourcing to a contract manufacturer., The bright spot was Apple, which this week also said that 10m people have signed up to the music-streaming service it launched six months ago., The biggest companies are building awe-inspiring headquarters: Apple’s “spaceship”, designed by Norman Foster, will cover 2.8m square feet (260,000 square metres) and Google’s new offices will sit under a vast, translucent dome., Apple’s iTunes store, historically the digital marketplace’s biggest music stall

In [25]:
for sentence in ent_sentences:
    indexes = calculate_entity_span(sentence, entity)
    calc_entity_subject_object(sentence, entity, indexes)


Sentence = Treading directly on the toes of Pandora, Apple will also offer individuals the chance to “create [their] own stations” based on their preferences.
Apple offer chance

Sentence = In 2014 Apple put a podcast app on the iPhone that is all but indelible.
Apple put app
Apple put on iPhone

Sentence = Nevertheless, Apple can now boast a new landmark: one billion active devices over the past 90 days.
Apple boast landmark
Apple boast over days

Sentence = Apple has also added haptic effects with something it calls a “taptic engine”, in effect a refined tiny vibrator which provides subtle taps in response to certain finger movements.
Apple added effects
Apple added vibrator

Sentence = On January 26th Apple announced profits for its most recent quarter of $18.4 billion, more than any listed firm worldwide has yet made in a three-month period.
Apple announced profits
Apple announced for quarter

Sentence = In its rise to greatness, Apple has repeatedly shrugged off bouts of panic am

In [27]:
entity = "CNN"

ent_sentences = return_docs_of_given_ent_type(processed_docs, entity, "ORG")
print(len(ent_sentences))

for sentence in ent_sentences:
    indexes = calculate_entity_span(sentence, entity)
    calc_entity_subject_object(sentence, entity, indexes)

8

Sentence = On CNN, Mr Sanders had this to say: "I am not greatly beloved by the economic establishment, by Wall Street, by the big money interests, or by the major media of this country including the Washington Post”.
 had On CNN


In [28]:
from spacy import displacy

text = "Last week, Democratic lawmakers from both parties said they had the Senate votes needed to pass legislation that would prevent tech platforms, including Apple, GM and Facebook, from favoring their own businesses."

doc = nlp(text)

displacy.render(doc, style="ent")


In [29]:
def visualize_given_ent_type(processed_docs, entity, ent_type):
    """
    Visualizes named entity annotations in sentences containing a given named entity of a specific type.

    Args:
        processed_docs (list): A list of processed documents. Each document is a spacy.tokens.Doc object.
        entity (str): The named entity to search for.
        ent_type (str): The type of the named entity (e.g. 'ORG', 'PERSON', 'GPE')

    Returns:
        None. Displays inline the named entity annotations for sentences containing the specified entity.
    """
    for doc in processed_docs:
        for sentence in doc.sents:
            if entity in [ent.text for ent in sentence.ents if ent.label_ == ent_type ]:
                displacy.render(sentence, style='ent' )

visualize_given_ent_type(processed_docs, 'Apple', 'ORG' )

#Now - Find sentences where the particular named entity is used alongside other entities of the same type - i.e. a specific entity type is used a particular number of times

In [30]:
def return_count_of_ent_type(sentence, ent_type):
    return len([ent.text for ent in sentence.ents if ent.label_ == ent_type ])

txt = "Last week, Democratic lawmakers from both parties said they had the Senate votes needed to pass legislation that would prevent tech platforms, including Apple, GM and Facebook, from favoring their own businesses."

doc = nlp(txt)

return_count_of_ent_type(doc, 'ORG')

4

In [31]:
def return_docs_of_given_ent_type_custom(processed_docs, entity, ent_type):
    """
    Returns sentences containing a given named entity of a specific type,
    but only if there is more than one occurrence of the entity type in the sentence.

    Args:
        processed_docs (list): A list of processed documents. Each document is a spacy.tokens.Doc object.
        entity (str): The named entity to search for.
        ent_type (str): The type of the named entity (e.g. 'ORG', 'PERSON', 'GPE')

    Returns:
        output_sentences (list): A list of sentences that contain the specified entity
                                 and have more than one occurrence of the specified entity type.
    """
    output_sentences = []
    for doc in processed_docs:
        for sentence in doc.sents:
            if entity in [ent.text for ent in sentence.ents if ent.label_ == ent_type and
                          return_count_of_ent_type(sentence, ent_type) > 1 ]:
                output_sentences.append(sentence)
    return output_sentences

output_sentences = return_docs_of_given_ent_type_custom(processed_docs, "Apple", "ORG")

print(len(output_sentences))


33


In [32]:
def visualize_conditional_sentences(sentences):
    """
    This function visualizes the named entities in the given sentences using the displaCy visualizer from SpaCy.
    It specifically highlights organizations (ORG) in the sentences.

    Parameters:
    sentences (list): A list of sentences. Each sentence is expected to be a SpaCy Doc object with named entities.

    Returns:
    None. The function directly renders the visualization in the Jupyter notebook or other environment.
    """
    colors = {"ORG": "linear-gradient(90deg, #64B5F6, #E0F7FA)"}
    options = {"ents": ["ORG"], "colors": colors}

    for sentence in sentences:
        displacy.render(sentence, style="ent", options=options)



visualize_conditional_sentences(output_sentences )